In [1]:
from pathlib import Path
import pandas as pd
import json

import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_DATASET_ENTRADA = "CIC18__cleanning__v1"
NOMBRE_SPLIT = "CIC18__split__v1"

RUTA_DATASET_LIMPIO = PROJECT_ROOT / "02_datasets" / "processed" / NOMBRE_DATASET_ENTRADA
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed" / NOMBRE_SPLIT

NOMBRE_DATASET_LIMPIO = f"{NOMBRE_DATASET_ENTRADA}.csv"
NOMBRE_TRAIN = f"{NOMBRE_SPLIT}__train.csv"
NOMBRE_TEST = f"{NOMBRE_SPLIT}__test.csv"

NOMBRE_REPORTE = f"{NOMBRE_SPLIT}_report.json"
RUTA_REPORTE = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_SPLIT

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"
TEST_SIZE = 0.20
RANDOM_STATE = 42

In [3]:
print("PROJECT_ROOT:")
print(PROJECT_ROOT)
print()

print("Dataset limpio de entrada:")
print(RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO)
print()

print("Ruta de salida:")
print(RUTA_SALIDA)

PROJECT_ROOT:
/home/javier/TFG_MODELOS_SIMPLES

Dataset limpio de entrada:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC18__cleanning__v1/CIC18__cleanning__v1.csv

Ruta de salida:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC18__split__v1


In [4]:
input_path = RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO

if not input_path.exists():
    raise FileNotFoundError(f"No existe el dataset limpio en: {input_path}")

df = cargar_dataset(nombre_dataset=NOMBRE_DATASET_LIMPIO, ruta_base=RUTA_DATASET_LIMPIO)

shape_original = df.shape

print("Forma del dataset limpio:")
print(shape_original)

Forma del dataset limpio:
(1676437, 55)


In [5]:
df.head()

,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,TOTLEN_BWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,...,INIT_BWD_WIN_BYTS,FWD_ACT_DATA_PKTS,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MEAN,IDLE_MAX,IDLE_MIN,LABEL
0,0,0,2743380,13,5,2012,329,215,0,61452,...,57,9,0,0,0,0,0,0,0,0
1,0,0,3793412,92,128,11409,1578,1072,0,125984,...,128,25,865369,717798,679356,165,1073758,65455,829881,0
2,50,0,1189116,26,5,2387,3605,484,0,21307,...,1522,6,0,0,0,0,0,0,0,0
3,2126,0,323,12,1,3,1,3,0,3,...,57,1,0,0,0,0,0,0,0,0
4,16,2,1359425,12,1,107,546,14,1,14,...,3,1,0,0,0,0,0,0,0,0


In [6]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Columna objetivo encontrada correctamente.")
print()
print("Distribución global de clases:")
display(resumen_clases(df, LABEL_COL))

Columna objetivo encontrada correctamente.

Distribución global de clases:


,count,percentage
LABEL,,
0,450000,26.8426
1,450000,26.8426
2,198861,11.8621
3,145199,8.6612
4,144535,8.6216
5,139775,8.3376
6,94048,5.6100
7,41406,2.4699
8,9908,0.5910


In [7]:
train_df, test_df = dividir_train_test_stratified(
    df=df,
    label_col=LABEL_COL,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Forma train:", train_df.shape)
print("Forma test:", test_df.shape)

Forma train: (1341149, 55)
Forma test: (335288, 55)


In [8]:
print("Distribución de clases en TRAIN:")
display(resumen_clases(train_df, LABEL_COL))

Distribución de clases en TRAIN:


,count,percentage
LABEL,,
0,360000,26.8427
1,360000,26.8427
2,159089,11.8621
3,116159,8.6612
4,115628,8.6216
5,111820,8.3376
6,75238,5.6100
7,33125,2.4699
8,7926,0.5910


In [9]:
print("Distribución de clases en TEST:")
display(resumen_clases(test_df, LABEL_COL))

Distribución de clases en TEST:


,count,percentage
LABEL,,
0,90000,26.8426
1,90000,26.8426
2,39772,11.8620
3,29040,8.6612
4,28907,8.6215
5,27955,8.3376
6,18810,5.6101
7,8281,2.4698
8,1982,0.5911


In [10]:
resumen_global = resumen_clases(df, LABEL_COL).rename(
    columns={"count": "global_count", "percentage": "global_percentage"}
)

resumen_train = resumen_clases(train_df, LABEL_COL).rename(
    columns={"count": "train_count", "percentage": "train_percentage"}
)

resumen_test = resumen_clases(test_df, LABEL_COL).rename(
    columns={"count": "test_count", "percentage": "test_percentage"}
)

comparacion = pd.concat([resumen_global, resumen_train, resumen_test], axis=1)

print("Comparación global / train / test:")
display(comparacion)

Comparación global / train / test:


,global_count,global_percentage,train_count,train_percentage,test_count,test_percentage
LABEL,,,,,,
0,450000,26.8426,360000,26.8427,90000,26.8426
1,450000,26.8426,360000,26.8427,90000,26.8426
2,198861,11.8621,159089,11.8621,39772,11.8620
3,145199,8.6612,116159,8.6612,29040,8.6612
4,144535,8.6216,115628,8.6216,28907,8.6215
5,139775,8.3376,111820,8.3376,27955,8.3376
6,94048,5.6100,75238,5.6100,18810,5.6101
7,41406,2.4699,33125,2.4699,8281,2.4698
8,9908,0.5910,7926,0.5910,1982,0.5911


In [11]:
guardar_dataset_csv(
    df=train_df,
    nombre_archivo=NOMBRE_TRAIN,
    ruta=RUTA_SALIDA
)

guardar_dataset_csv(
    df=test_df,
    nombre_archivo=NOMBRE_TEST,
    ruta=RUTA_SALIDA
)

print("Train guardado en:")
print(RUTA_SALIDA / NOMBRE_TRAIN)
print()
print("Test guardado en:")
print(RUTA_SALIDA / NOMBRE_TEST)

Train guardado en:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__train.csv

Test guardado en:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__test.csv


In [12]:
reporte_split = {
    "dataset_entrada": NOMBRE_DATASET_LIMPIO,
    "label_column": LABEL_COL,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "shape_global": {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1])
    },
    "shape_train": {
        "rows": int(train_df.shape[0]),
        "cols": int(train_df.shape[1])
    },
    "shape_test": {
        "rows": int(test_df.shape[0]),
        "cols": int(test_df.shape[1])
    },
    "class_distribution_global": {
        str(k): int(v) for k, v in df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_train": {
        str(k): int(v) for k, v in train_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_test": {
        str(k): int(v) for k, v in test_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    }
}

report_path = RUTA_REPORTE / NOMBRE_REPORTE

report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(reporte_split, f, indent=2, ensure_ascii=False)

print("Reporte guardado en:")
print(report_path)

Reporte guardado en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC18__split__v1/CIC18__split__v1_report.json


In [13]:
print("========== RESUMEN SPLIT ==========")
print(f"Dataset de entrada: {NOMBRE_DATASET_LIMPIO}")
print(f"Forma global: {df.shape}")
print(f"Forma train: {train_df.shape}")
print(f"Forma test: {test_df.shape}")
print(f"Test size: {TEST_SIZE}")
print(f"Random state: {RANDOM_STATE}")
print("===================================")

========== RESUMEN SPLIT ==========
Dataset de entrada: CIC18__cleanning__v1.csv
Forma global: (1676437, 55)
Forma train: (1341149, 55)
Forma test: (335288, 55)
Test size: 0.2
Random state: 42
